# Day 5 — Preprocessing & Train/Test Split

Goal: prepare the AI4I 2020 dataset for modeling.

Steps:
1. Load the dataset
2. Drop columns that cause target leakage
3. Separate features (X) from target (y)
4. One-hot encode the categorical `Type` column
5. Stratified 80/20 train/test split
6. Scale numeric features with `StandardScaler` (fit on train only)
7. Save artifacts to `../models/` for Day 6 and Day 7

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

## 1. Load the dataset

In [2]:
df = pd.read_csv("../data/ai4i2020.csv")
print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 2. Drop columns that cause target leakage

- `UDI`, `Product ID` are identifiers with no predictive value
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF` are specific failure-type flags computed *after* a failure occurs — using them would let the model cheat

In [3]:
cols_to_drop = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]
df = df.drop(columns=cols_to_drop)
print("Shape after drop:", df.shape)
df.head()

Shape after drop: (10000, 7)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


## 3. Separate features (X) and target (y)

In [4]:
X = df.drop(columns=["Machine failure"])
y = df["Machine failure"]
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Failure rate:", y.mean().round(4))

X shape: (10000, 6)
y shape: (10000,)
Failure rate: 0.0339


## 4. One-hot encode the Type column

ML algorithms need numbers, not text like "L", "M", "H". `drop_first=True` avoids the dummy variable trap — the third column is redundant since it can be derived from the other two.

In [5]:
X = pd.get_dummies(X, columns=["Type"], drop_first=True)
print("X shape after one-hot:", X.shape)
print("Columns:", list(X.columns))
X.head()

X shape after one-hot: (10000, 7)
Columns: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_L', 'Type_M']


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_L,Type_M
0,298.1,308.6,1551,42.8,0,False,True
1,298.2,308.7,1408,46.3,3,True,False
2,298.1,308.5,1498,49.4,5,True,False
3,298.2,308.6,1433,39.5,7,True,False
4,298.2,308.7,1408,40.0,9,True,False


## 5. Stratified 80/20 train/test split

- `test_size=0.20` → 20% held out for final evaluation
- `random_state=42` → reproducibility
- `stratify=y` → preserves the ~3.4% failure rate in BOTH sets (critical for imbalanced data)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("y_train:", y_train.shape, "  y_test:", y_test.shape)
print("Train failure rate:", y_train.mean().round(4))
print("Test failure rate: ", y_test.mean().round(4))

X_train: (8000, 7)   X_test: (2000, 7)
y_train: (8000,)   y_test: (2000,)
Train failure rate: 0.0339
Test failure rate:  0.034


## 6. Scale numeric features

Logistic Regression weights features by magnitude — torque (~40) would be dwarfed by rotational speed (~1500) without scaling.

**Critical rule:** fit scaler on training data only. Using `transform` (not `fit_transform`) on the test set prevents data leakage.

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape: ", X_test_scaled.shape)

# Verify scaling worked
scaled_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
print("\nMeans (should be ~0):")
print(scaled_train_df.mean().round(4))
print("\nStandard deviations (should be ~1):")
print(scaled_train_df.std().round(4))

X_train_scaled shape: (8000, 7)
X_test_scaled shape:  (2000, 7)

Means (should be ~0):
Air temperature [K]        0.0
Process temperature [K]    0.0
Rotational speed [rpm]     0.0
Torque [Nm]               -0.0
Tool wear [min]           -0.0
Type_L                    -0.0
Type_M                     0.0
dtype: float64

Standard deviations (should be ~1):
Air temperature [K]        1.0001
Process temperature [K]    1.0001
Rotational speed [rpm]     1.0001
Torque [Nm]                1.0001
Tool wear [min]            1.0001
Type_L                     1.0001
Type_M                     1.0001
dtype: float64


## 7. Save artifacts to `../models/`

Save the scaled data, the labels, the fitted scaler, and the feature names so Day 6 and Day 7 can load them.

In [8]:
os.makedirs("../models", exist_ok=True)

joblib.dump(X_train_scaled, "../models/X_train_scaled.pkl")
joblib.dump(X_test_scaled,  "../models/X_test_scaled.pkl")
joblib.dump(y_train,        "../models/y_train.pkl")
joblib.dump(y_test,         "../models/y_test.pkl")
joblib.dump(scaler,         "../models/scaler.pkl")
joblib.dump(list(X_train.columns), "../models/feature_names.pkl")

print("All files saved to ../models/")
print("\nFolder contents:")
for f in sorted(os.listdir("../models")):
    print(" ", f)

All files saved to ../models/

Folder contents:
  X_test_scaled.pkl
  X_train_scaled.pkl
  feature_names.pkl
  scaler.pkl
  y_test.pkl
  y_train.pkl


## Day 5 summary

### What I did
1. Dropped target-leakage columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) and unused identifiers (`UDI`, `Product ID`)
2. Separated features (X) from target (y)
3. One-hot encoded the categorical `Type` column with `drop_first=True`
4. Performed a stratified 80/20 train/test split preserving the ~3.4% failure rate
5. Scaled numeric features with `StandardScaler` — fit on training data only
6. Saved 6 artifacts to `../models/` for use in Day 6 and Day 7

### Key concepts
| Concept | Summary |
|---|---|
| Target leakage | Features that contain information about the outcome — must be removed |
| One-hot encoding | Converting categorical text into 0/1 columns |
| Dummy variable trap | Redundant categorical columns — solved with `drop_first=True` |
| Train/test split | Holding out data to measure true generalization performance |
| Stratification | Preserving class proportions across splits |
| Feature scaling | Standardizing features so no single one dominates by magnitude |
| Data leakage | Fitting preprocessing on combined train+test inflates results |

### Next: Day 6
Train a Logistic Regression baseline on the scaled training data.